# Лабораторная работа №3: Foundation models для сегментации: SAM/SAM 2

**Версия:** студенческая  
**Курс:** «Современное компьютерное зрение»

> Сначала выполните notebook в режиме `FAST_DEV_RUN=True`, затем запускайте полный эксперимент. Сохраняйте метрики и checkpoints в `artifacts/`.


## Цели обучения

После работы студент должен уметь:

- Получить маски по point и box prompts.
- Автоматически сформировать prompts из GT или детектора.
- Сравнить zero-shot и refinement head.


## Теоретическая часть

Promptable segmentation оценивает маску $M=f(I,p)$ по изображению и prompt $p$ (точка, box или маска). SAM разделяет тяжёлый image encoder и лёгкий prompt/mask decoder, поэтому один embedding изображения можно использовать для многих запросов.

![SAM architecture](https://raw.githubusercontent.com/facebookresearch/segment-anything/main/assets/model_diagram.png)


### Математическая постановка

Качество promptable-модели оценивают не только IoU, но и числом взаимодействий до достижения целевого качества: $N@IoU\geq\tau$. Для ускорения разметки важна доля масок, принимаемых без существенной коррекции.


### Материалы

- [Segment Anything](https://github.com/facebookresearch/segment-anything)
- [Transformers SAM docs](https://huggingface.co/docs/transformers/model_doc/sam)

Ссылки даны как дополнительный материал. При изменении API сверяйтесь с актуальной официальной документацией и фиксируйте версии зависимостей.


## Практическое задание

1. Получить маски по point и box prompts.
2. Автоматически сформировать prompts из GT или детектора.
3. Сравнить zero-shot и refinement head.

**Обязательные артефакты:** код, конфигурация, метрики, минимум одна контролируемая ablation, примеры ошибок и краткие выводы.


### Профиль вычислений

- **Основной режим:** SAM ViT-B/MobileSAM inference, frozen encoder.
- **Ограничение данных:** до 200 изображений 512 px.
- **Fallback:** CPU на 20 изображениях или MobileSAM.
- Все длительные этапы должны сохранять checkpoint и продолжаться после перезапуска. Оценка не зависит от размера модели: важны корректность эксперимента и выводы.


**Оценивание.** Десять обязательных предметных этапов по 1 баллу: основная часть — 10 баллов. Творческий бонус — отдельно 1 балл, не заменяет обязательные этапы. Полный режим оценивается по реальным данным и обучению; `FAST_DEV_RUN=True` служит только smoke-проверкой, его показатели не являются отчётными.


## Реализация


In [55]:
# Базовое воспроизводимое окружение
from pathlib import Path
import json, os, random, time
import numpy as np
import pandas as pd

SEED = 42
FAST_DEV_RUN = False  # True — только проверка связности на уменьшенной выборке; не отчётный результат
ARTIFACTS = Path(os.getenv("CV_ARTIFACTS", "artifacts"))
ARTIFACTS.mkdir(exist_ok=True)
random.seed(SEED); np.random.seed(SEED)
try:
    import torch
    torch.manual_seed(SEED)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
except ImportError:
    torch = None
    DEVICE = "cpu"
# Датасеты курса лежат в общем каталоге <корень курса>/data:
# notebook выполняется из labs/lab_XX_*/, поэтому корень курса — cwd.parent.parent.
# Для локальных запусков вне labs/ также поднимаемся, пока не найдём каталог data с CIFAR.
_CANDIDATES = [Path.cwd().parent.parent / "data", Path.cwd().parent / "data", Path.cwd() / "data"]
DATA_ROOT = Path(os.getenv("CV_COURSE_DATA") or next((c for c in _CANDIDATES if (c / "cifar-10-batches-py").exists()), _CANDIDATES[0]))
DATA_ROOT.mkdir(parents=True, exist_ok=True)
print({"device": DEVICE, "fast_dev_run": FAST_DEV_RUN, "data_root": str(DATA_ROOT)})


{'device': 'cuda', 'fast_dev_run': False, 'data_root': 'c:\\Users\\Lenovo\\Projects\\data'}


In [56]:
def check(condition, message):
    """Мини-проверка в стиле YAAM: молчит, если всё хорошо; печатает, если найдена проблема."""
    if condition:
        print(f"✔ {message}")
    else:
        print(f"✘ ВНИМАНИЕ: {message}")

### Загрузка SAM и отложенных масок

*Вес: 1 балл.*

**Постановка.** Сравнение prompts возможно лишь на одних размеченных изображениях.

Выполните следующие действия:
1. Загрузите pretrained SAM и test Oxford-IIIT Pet.
2. Зафиксируйте индексы проверки.

**Результат.** dataset_manifest.json и объекты processor, model, eval_ds.

**Критерий принятия.** Задание считается принятым, если описанные действия выполнены, результат соответствует указанному артефакту и получен на предусмотренных данных; в режиме FAST_DEV_RUN результат служит только проверкой исполнения.


In [57]:
# Загружаем SAM ViT-B и Oxford-IIIT Pet.
# Фиксируем подмножество изображений для оценки (test) и обучения (trainval),
# чтобы уложиться в бюджет до 200 изображений 512 px.
from transformers import SamModel, SamProcessor
from torchvision.datasets import OxfordIIITPet
from torch.utils.data import Subset
from PIL import Image

N_EVAL = 200 if not FAST_DEV_RUN else 20
N_TRAIN = 200 if not FAST_DEV_RUN else 20

# SAM
processor = SamProcessor.from_pretrained("facebook/sam-vit-base")
model = SamModel.from_pretrained("facebook/sam-vit-base").to(DEVICE)
model.eval()

# Датасет: trimap (1=pet, 2=background, 3=outline) -> бинарная маска pet vs rest
def to_binary_mask(trimap: Image.Image) -> np.ndarray:
    arr = np.array(trimap)
    return (arr == 1).astype(np.uint8)  # 1 — питомец, 0 — всё остальное

class PetSegDataset(OxfordIIITPet):
    def __getitem__(self, idx):
        img, trimap = super().__getitem__(idx)
        return img, to_binary_mask(trimap)

test_ds_full = PetSegDataset(root=str(DATA_ROOT), split="test", target_types="segmentation", download=True)
trainval_ds_full = PetSegDataset(root=str(DATA_ROOT), split="trainval", target_types="segmentation", download=True)

rng = np.random.default_rng(SEED)
test_indices = rng.choice(len(test_ds_full), size=min(N_EVAL, len(test_ds_full)), replace=False).tolist()
train_indices = rng.choice(len(trainval_ds_full), size=min(N_TRAIN, len(trainval_ds_full)), replace=False).tolist()

eval_ds = Subset(test_ds_full, test_indices)
train_ds = Subset(trainval_ds_full, train_indices)

manifest = {
    "sam_checkpoint": "facebook/sam-vit-base",
    "dataset": "OxfordIIITPet",
    "test_indices": test_indices,
    "train_indices": train_indices,
    "n_eval": len(eval_ds),
    "n_train": len(train_ds),
}
with open(ARTIFACTS / "dataset_manifest.json", "w") as f:
    json.dump(manifest, f, indent=2)

print(f"✔ SAM и датасет загружены. eval={len(eval_ds)}, train={len(train_ds)}")
print(f"✔ Манифест сохранён: {ARTIFACTS / 'dataset_manifest.json'}")

Loading weights:   0%|          | 0/314 [00:00<?, ?it/s]

✔ SAM и датасет загружены. eval=200, train=200
✔ Манифест сохранён: artifacts\dataset_manifest.json


### Геометрия подсказок и IoU

*Вес: 1 балл.*

**Постановка.** Box и point должны воспроизводимо извлекаться из одной истинной маски.

Выполните следующие действия:
1. Реализуйте IoU масок и ограничивающий прямоугольник.
2. Документируйте, что подсказки из GT моделируют действия разметчика.

**Результат.** Функции iou и box_from_mask.

**Критерий принятия.** Задание считается принятым, если описанные действия выполнены, результат соответствует указанному артефакту и получен на предусмотренных данных; в режиме FAST_DEV_RUN результат служит только проверкой исполнения.


In [58]:
def iou(mask_true: np.ndarray, mask_pred: np.ndarray) -> float:
    """IoU двух бинарных масок (bool или 0/1)."""
    m1 = mask_true.astype(bool).ravel()
    m2 = mask_pred.astype(bool).ravel()
    inter = np.logical_and(m1, m2).sum()
    union = np.logical_or(m1, m2).sum()
    return float(inter / union) if union > 0 else 0.0

def box_from_mask(mask: np.ndarray) -> tuple:
    """Возвращает (x_min, y_min, x_max, y_max) по ненулевым пикселям маски."""
    ys, xs = np.where(mask > 0)
    if len(xs) == 0:
        return (0, 0, 1, 1)
    return (int(xs.min()), int(ys.min()), int(xs.max()), int(ys.max()))

# Демонстрация на одном примере: подсказки из GT моделируют действия разметчика
_img, _mask = eval_ds[0]
print(f"✔ IoU идентичных масок = {iou(_mask, _mask):.3f}")
print(f"✔ box_from_mask: {box_from_mask(_mask)}")


✔ IoU идентичных масок = 1.000
✔ box_from_mask: (0, 16, 472, 439)


### Предсказание SAM по подсказке

*Вес: 1 балл.*

**Постановка.** Encoder и mask decoder должны получать корректный тип prompt.

Выполните следующие действия:
1. Подайте point или box в processor.
2. Выберите маску по предсказанной SAM оценке IoU.

**Результат.** Функция predict_sam с бинарной маской исходного размера.

**Критерий принятия.** Задание считается принятым, если описанные действия выполнены, результат соответствует указанному артефакту и получен на предусмотренных данных; в режиме FAST_DEV_RUN результат служит только проверкой исполнения.


In [59]:
@torch.no_grad()
def predict_sam(image: Image.Image, prompt: dict, processor, model, device=DEVICE) -> np.ndarray:
    """
    prompt: {'type': 'point', 'coords': [[x, y]]} или
            {'type': 'box', 'coords': [[x_min, y_min, x_max, y_max]]}
    Возвращает бинарную маску исходного размера (uint8).
    """
    if prompt["type"] == "point":
        inputs = processor(image, input_points=[prompt["coords"]], return_tensors="pt").to(device)
    elif prompt["type"] == "box":
        inputs = processor(image, input_boxes=[prompt["coords"]], return_tensors="pt").to(device)
    else:
        raise ValueError(f"Неизвестный тип prompt: {prompt['type']}")

    outputs = model(**inputs)
    masks_processed = processor.image_processor.post_process_masks(
        outputs.pred_masks.cpu(),
        inputs["original_sizes"].cpu(),
        inputs["reshaped_input_sizes"].cpu(),
    )
    # masks_processed[0] может быть (num_points, num_masks, H, W) или (num_masks, H, W).
    # Приводим к строго 3D: (num_masks, H, W).
    mask_tensor = masks_processed[0]
    while mask_tensor.dim() > 3:
        mask_tensor = mask_tensor[0]

    num_masks = mask_tensor.shape[0]
    # iou_scores для batch=1, num_points=1 имеет форму (1, 1, num_masks) или (1, num_masks).
    scores = outputs.iou_scores[0].reshape(-1)
    if scores.numel() > num_masks:
        scores = scores[:num_masks]

    best_idx = int(torch.argmax(scores))
    best_mask = (mask_tensor[best_idx].numpy() > 0).astype(np.uint8)
    return best_mask

# Быстрая проверка на одном изображении
_img, _gt = eval_ds[0]
_x0, _y0, _x1, _y1 = box_from_mask(_gt)
_pt_mask = predict_sam(_img, {"type": "point", "coords": [[int((_x0+_x1)/2), int((_y0+_y1)/2)]]}, processor, model)
_box_mask = predict_sam(_img, {"type": "box", "coords": [[_x0, _y0, _x1, _y1]]}, processor, model)
print(f"✔ shapes: gt={_gt.shape}, point={_pt_mask.shape}, box={_box_mask.shape}")
print(f"✔ SAM point IoU = {iou(_gt, _pt_mask):.3f}, box IoU = {iou(_gt, _box_mask):.3f}")

✔ shapes: gt=(440, 500), point=(440, 500), box=(440, 500)
✔ SAM point IoU = 0.430, box IoU = 0.674


### Сравнение point и box

*Вес: 1 балл.*

**Постановка.** Степень интерактивной подсказки влияет на качество сегментации.

Выполните следующие действия:
1. Получите две маски для каждого отложенного изображения.
2. Сохраните IoU, площади и координаты prompts.

**Результат.** prompt_results.json с реальными оценками SAM.

**Критерий принятия.** Задание считается принятым, если описанные действия выполнены, результат соответствует указанному артефакту и получен на предусмотренных данных; в режиме FAST_DEV_RUN результат служит только проверкой исполнения.


In [60]:
prompt_results = []
for i in range(len(eval_ds)):
    img, gt = eval_ds[i]
    x0, y0, x1, y1 = box_from_mask(gt)
    cx, cy = int((x0 + x1) / 2), int((y0 + y1) / 2)

    pt_mask = predict_sam(img, {"type": "point", "coords": [[cx, cy]]}, processor, model)
    box_mask = predict_sam(img, {"type": "box", "coords": [[x0, y0, x1, y1]]}, processor, model)

    prompt_results.append({
        "index": i,
        "point_iou": iou(gt, pt_mask),
        "box_iou": iou(gt, box_mask),
        "point_area": int(pt_mask.sum()),
        "box_area": int(box_mask.sum()),
        "gt_area": int(gt.sum()),
        "point_prompt": [cx, cy],
        "box_prompt": [x0, y0, x1, y1],
    })

with open(ARTIFACTS / "prompt_results.json", "w") as f:
    json.dump(prompt_results, f, indent=2)

df_prompts = pd.DataFrame(prompt_results)
display(df_prompts[["index", "point_iou", "box_iou", "gt_area", "point_area", "box_area"]].head(10))
print(f"✔ Средний IoU point = {df_prompts['point_iou'].mean():.4f}")
print(f"✔ Средний IoU box   = {df_prompts['box_iou'].mean():.4f}")
print(f"✔ Результаты сохранены: {ARTIFACTS / 'prompt_results.json'}")

,index,point_iou,box_iou,gt_area,point_area,box_area
0,0,0.430133,0.673968,130475,62224,177269
1,1,0.771043,0.766703,44466,54066,57842
2,2,0.804193,0.808259,80198,99597,99033
3,3,0.719827,0.718038,29955,41607,40433
4,4,0.205922,0.669171,6417,2543,9582
5,5,0.731355,0.724714,46200,62169,63616
6,6,0.720131,0.727689,9222,12806,12673
7,7,0.862306,0.854665,80661,91580,94067
8,8,0.902271,0.900403,159416,176255,176771
9,9,0.687491,0.697178,9959,14486,14275


✔ Средний IoU point = 0.6860
✔ Средний IoU box   = 0.8002
✔ Результаты сохранены: artifacts\prompt_results.json


### Обучение supervised baseline

*Вес: 1 балл.*

**Постановка.** Zero-shot SAM следует сравнивать с обученной на отдельных изображениях моделью.

Выполните следующие действия:
1. Подготовьте бинарные маски trainval.
2. Обучите LRASPP без доступа к test и сохраните веса.

**Результат.** supervised.pt с обученными весами.

**Критерий принятия.** Задание считается принятым, если описанные действия выполнены, результат соответствует указанному артефакту и получен на предусмотренных данных; в режиме FAST_DEV_RUN результат служит только проверкой исполнения.


In [61]:
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import transforms
from torchvision.models.segmentation import lraspp_mobilenet_v3_large

# Приводим изображения и маски к 512×512 для обучения baseline и корректора
train_tf = transforms.Compose([
    transforms.Resize((512, 512)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])
mask_tf = transforms.Compose([
    transforms.Resize((512, 512), interpolation=transforms.InterpolationMode.NEAREST),
    transforms.PILToTensor(),
])

class PetSegTrainDataset(PetSegDataset):
    def __getitem__(self, idx):
        img, trimap = super().__getitem__(idx)
        img_t = train_tf(img)
        mask_t = mask_tf(Image.fromarray(trimap)).squeeze(0).long()  # (H, W)
        return img_t, mask_t

train_seg_ds = Subset(
    PetSegTrainDataset(root=str(DATA_ROOT), split="trainval", target_types="segmentation"),
    train_indices
)
train_loader = DataLoader(train_seg_ds, batch_size=8, shuffle=True, num_workers=0, pin_memory=torch.cuda.is_available())

# Загружаем предобученный LRASPP (голова на 21 класс — как в чекпоинте)
torch.manual_seed(SEED)
supervised = lraspp_mobilenet_v3_large(weights="DEFAULT").to(DEVICE)

# Заменяем ТОЛЬКО последние свёрточные слои внутри классификатора под 2 класса.
# Для MobileNetV3-Large: low_channels=40, inter_channels=128.
supervised.classifier.low_classifier = nn.Conv2d(40, 2, 1).to(DEVICE)
supervised.classifier.high_classifier = nn.Conv2d(128, 2, 1).to(DEVICE)

opt = torch.optim.AdamW(supervised.parameters(), lr=1e-4)
criterion = nn.CrossEntropyLoss()

EPOCHS = 5 if not FAST_DEV_RUN else 1
supervised.train()
for epoch in range(EPOCHS):
    total_loss = 0.0
    for imgs, masks in train_loader:
        imgs, masks = imgs.to(DEVICE), masks.to(DEVICE)
        opt.zero_grad()
        out = supervised(imgs)["out"]  # (B, 2, H, W)
        loss = criterion(out, masks)
        loss.backward()
        opt.step()
        total_loss += loss.item() * imgs.size(0)
    print(f"✔ Supervised epoch {epoch+1}/{EPOCHS}, loss={total_loss/len(train_seg_ds):.4f}")

torch.save(supervised.state_dict(), ARTIFACTS / "supervised.pt")
print(f"✔ Веса supervised сохранены: {ARTIFACTS / 'supervised.pt'}")

✔ Supervised epoch 1/5, loss=0.5176
✔ Supervised epoch 2/5, loss=0.2533
✔ Supervised epoch 3/5, loss=0.1774
✔ Supervised epoch 4/5, loss=0.1459
✔ Supervised epoch 5/5, loss=0.1266
✔ Веса supervised сохранены: artifacts\supervised.pt


### Парное сравнение с baseline

*Вес: 1 балл.*

**Постановка.** Парные изображения исключают разницу в составе оценочной выборки.

Выполните следующие действия:
1. Предскажите test-маски supervised модели.
2. Сравните её IoU с SAM box на тех же индексах.

**Результат.** supervised_comparison.json с парными IoU.

**Критерий принятия.** Задание считается принятым, если описанные действия выполнены, результат соответствует указанному артефакту и получен на предусмотренных данных; в режиме FAST_DEV_RUN результат служит только проверкой исполнения.


In [62]:
@torch.no_grad()
def predict_supervised(model, image: Image.Image, size=(512, 512)) -> np.ndarray:
    model.eval()
    img_t = train_tf(image).unsqueeze(0).to(DEVICE)
    out = model(img_t)["out"]
    pred = out.argmax(dim=1).squeeze(0).cpu().numpy().astype(np.uint8)
    # Возвращаем к исходному размеру для честного сравнения с GT
    pred_pil = Image.fromarray(pred)
    pred_pil = pred_pil.resize(image.size, resample=Image.NEAREST)
    return np.array(pred_pil)

supervised_comparison = []
for i in range(len(eval_ds)):
    img, gt = eval_ds[i]
    x0, y0, x1, y1 = box_from_mask(gt)
    sam_box = predict_sam(img, {"type": "box", "coords": [[x0, y0, x1, y1]]}, processor, model)
    sup_mask = predict_supervised(supervised, img)

    supervised_comparison.append({
        "index": i,
        "sam_box_iou": iou(gt, sam_box),
        "supervised_iou": iou(gt, sup_mask),
    })

with open(ARTIFACTS / "supervised_comparison.json", "w") as f:
    json.dump(supervised_comparison, f, indent=2)

df_sup = pd.DataFrame(supervised_comparison)
display(df_sup.head(10))
print(f"✔ Средний IoU SAM box      = {df_sup['sam_box_iou'].mean():.4f}")
print(f"✔ Средний IoU supervised  = {df_sup['supervised_iou'].mean():.4f}")
print(f"✔ Парное сравнение сохранено: {ARTIFACTS / 'supervised_comparison.json'}")

,index,sam_box_iou,supervised_iou
0,0,0.673968,0.918224
1,1,0.766703,0.831026
2,2,0.808259,0.847548
3,3,0.718038,0.812406
4,4,0.669171,0.640487
5,5,0.724714,0.799285
6,6,0.727689,0.793208
7,7,0.854665,0.825744
8,8,0.900403,0.943846
9,9,0.697178,0.752080


✔ Средний IoU SAM box      = 0.8002
✔ Средний IoU supervised  = 0.7828
✔ Парное сравнение сохранено: artifacts\supervised_comparison.json


### Обучающие маски для корректора

*Вес: 1 балл.*

**Постановка.** Корректор должен обучаться только на train-изображениях и масках SAM.

Выполните следующие действия:
1. Получите SAM-предсказания для trainval.
2. Сформируйте четырёхканальные входы из RGB и маски.

**Результат.** refine_train из реальных изображений и масок.

**Критерий принятия.** Задание считается принятым, если описанные действия выполнены, результат соответствует указанному артефакту и получен на предусмотренных данных; в режиме FAST_DEV_RUN результат служит только проверкой исполнения.


In [63]:
# Генерируем SAM-предсказания на train-изображениях (box prompt из GT)
# и формируем 4-канальные входы: RGB + SAM-маска.
# ВАЖНО: метки и канал маски — 0/1 float (без /255), иначе BCE по факту
# учит голову предсказывать нули, и на test получаются пустые маски.
refine_train = []
for i in range(len(train_ds)):
    img, gt = train_ds[i]
    x0, y0, x1, y1 = box_from_mask(gt)
    sam_mask = predict_sam(img, {"type": "box", "coords": [[x0, y0, x1, y1]]}, processor, model)

    # Приводим к 512×512 для обучения головы
    img_resized = img.resize((512, 512), Image.BILINEAR)
    sam_resized = Image.fromarray(sam_mask).resize((512, 512), Image.NEAREST)
    gt_resized = Image.fromarray(gt).resize((512, 512), Image.NEAREST)

    img_t = transforms.ToTensor()(img_resized)                                       # (3, 512, 512) в [0,1]
    sam_t = torch.from_numpy(np.array(sam_resized)).float().unsqueeze(0)             # 0/1
    gt_t = torch.from_numpy(np.array(gt_resized)).float().unsqueeze(0)               # 0/1 — БЕЗ /255

    refine_train.append((torch.cat([img_t, sam_t], dim=0), gt_t))  # (4,H,W), (1,H,W)

print(f"✔ Сформировано {len(refine_train)} обучающих примеров для корректора (4-канальный вход).")
# Диагностика: значения метки и SAM-канала
print(f"✔ label range: [{refine_train[0][1].min().item():.3f}, {refine_train[0][1].max().item():.3f}]")
print(f"✔ sam   range: [{refine_train[0][0][3].min().item():.3f}, {refine_train[0][0][3].max().item():.3f}]")

✔ Сформировано 200 обучающих примеров для корректора (4-канальный вход).
✔ label range: [0.000, 1.000]
✔ sam   range: [0.000, 1.000]


### Обучение корректирующей головы

*Вес: 1 балл.*

**Постановка.** Обучаемая свёрточная голова может уточнять ошибочную маску.

Выполните следующие действия:
1. Обучите голову на подготовленных четырёхканальных входах.
2. Сохраните веса для повторного использования.

**Результат.** refinement.pt после фактической оптимизации.

**Критерий принятия.** Задание считается принятым, если описанные действия выполнены, результат соответствует указанному артефакту и получен на предусмотренных данных; в режиме FAST_DEV_RUN результат служит только проверкой исполнения.


In [64]:
class RefineHead(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(4, 32, 3, padding=1), nn.ReLU(inplace=True),
            nn.Conv2d(32, 32, 3, padding=1), nn.ReLU(inplace=True),
            nn.Conv2d(32, 1, 1),
        )

    def forward(self, x):
        return self.net(x)

refine_loader = DataLoader(refine_train, batch_size=8, shuffle=True, num_workers=0)
torch.manual_seed(SEED)
refine_head = RefineHead().to(DEVICE)
opt_ref = torch.optim.AdamW(refine_head.parameters(), lr=1e-3)
bce = nn.BCEWithLogitsLoss()

EPOCHS_REF = 10 if not FAST_DEV_RUN else 1
refine_head.train()
for epoch in range(EPOCHS_REF):
    total_loss = 0.0
    for x, y in refine_loader:
        x, y = x.to(DEVICE), y.to(DEVICE)
        opt_ref.zero_grad()
        logits = refine_head(x)
        loss = bce(logits, y)
        loss.backward()
        opt_ref.step()
        total_loss += loss.item() * x.size(0)
    print(f"✔ Refinement epoch {epoch+1}/{EPOCHS_REF}, loss={total_loss/len(refine_train):.4f}")

torch.save(refine_head.state_dict(), ARTIFACTS / "refinement.pt")
print(f"✔ Веса корректора сохранены: {ARTIFACTS / 'refinement.pt'}")

✔ Refinement epoch 1/10, loss=0.5657
✔ Refinement epoch 2/10, loss=0.2977
✔ Refinement epoch 3/10, loss=0.2413
✔ Refinement epoch 4/10, loss=0.2312
✔ Refinement epoch 5/10, loss=0.2263
✔ Refinement epoch 6/10, loss=0.2251
✔ Refinement epoch 7/10, loss=0.2170
✔ Refinement epoch 8/10, loss=0.2186
✔ Refinement epoch 9/10, loss=0.2135
✔ Refinement epoch 10/10, loss=0.2139
✔ Веса корректора сохранены: artifacts\refinement.pt


### Независимая оценка коррекции

*Вес: 1 балл.*

**Постановка.** Качество коррекции необходимо измерить на отложенных изображениях.

Выполните следующие действия:
1. Примените обученную голову к SAM-маскам test.
2. Сохраните IoU и долю исходных масок, требующих существенной правки.

**Результат.** adapted_comparison.json с измеренными IoU.

**Критерий принятия.** Задание считается принятым, если описанные действия выполнены, результат соответствует указанному артефакту и получен на предусмотренных данных; в режиме FAST_DEV_RUN результат служит только проверкой исполнения.


In [65]:
refine_head.eval()

@torch.no_grad()
def apply_refinement(image: Image.Image, sam_mask: np.ndarray, refine_head, size=512) -> np.ndarray:
    """Применяет обученную голову к SAM-маске и возвращает бинарную маску исходного размера."""
    img_r = image.resize((size, size), Image.BILINEAR)
    sam_r = Image.fromarray(sam_mask).resize((size, size), Image.NEAREST)
    img_t = transforms.ToTensor()(img_r)
    sam_t = torch.from_numpy(np.array(sam_r)).float().unsqueeze(0)   # 0/1 — как при обучении
    x = torch.cat([img_t, sam_t], dim=0).unsqueeze(0).to(DEVICE)
    logits = refine_head(x)
    pred = (torch.sigmoid(logits) > 0.5).squeeze().cpu().numpy().astype(np.uint8)
    pred_pil = Image.fromarray(pred).resize(image.size, Image.NEAREST)
    return np.array(pred_pil)

adapted_comparison = []
for i in range(len(eval_ds)):
    img, gt = eval_ds[i]
    x0, y0, x1, y1 = box_from_mask(gt)
    sam_mask = predict_sam(img, {"type": "box", "coords": [[x0, y0, x1, y1]]}, processor, model)
    corrected = apply_refinement(img, sam_mask, refine_head)

    sam_iou = iou(gt, sam_mask)
    corrected_iou = iou(gt, corrected)
    adapted_comparison.append({
        "index": i,
        "sam_iou": sam_iou,
        "corrected_iou": corrected_iou,
        "improved": bool(corrected_iou > sam_iou),
        "need_major_fix": bool(sam_iou < 0.7),
    })

with open(ARTIFACTS / "adapted_comparison.json", "w") as f:
    json.dump(adapted_comparison, f, indent=2)

df_adapt = pd.DataFrame(adapted_comparison)
display(df_adapt.head(10))
print(f"✔ Средний IoU SAM (box)   = {df_adapt['sam_iou'].mean():.4f}")
print(f"✔ Средний IoU corrected   = {df_adapt['corrected_iou'].mean():.4f}")
print(f"✔ Доля improved           = {df_adapt['improved'].mean():.2%}")
print(f"✔ Доля масок с IoU < 0.7 = {df_adapt['need_major_fix'].mean():.2%}")
print(f"✔ Результаты сохранены: {ARTIFACTS / 'adapted_comparison.json'}")

,index,sam_iou,corrected_iou,improved,need_major_fix
0,0,0.673968,0.675279,True,True
1,1,0.766703,0.781651,True,False
2,2,0.808259,0.816229,True,False
3,3,0.718038,0.729603,True,False
4,4,0.669171,0.694141,True,True
5,5,0.724714,0.740611,True,False
6,6,0.727689,0.748053,True,False
7,7,0.854665,0.862358,True,False
8,8,0.900403,0.907023,True,False
9,9,0.697178,0.736141,True,True


✔ Средний IoU SAM (box)   = 0.8002
✔ Средний IoU corrected   = 0.8130
✔ Доля improved           = 99.00%
✔ Доля масок с IoU < 0.7 = 15.00%
✔ Результаты сохранены: artifacts\adapted_comparison.json


### Сопоставление трудных случаев

*Вес: 1 балл.*

**Постановка.** Средний IoU маскирует случаи отрицательного переноса после уточнения.

Выполните следующие действия:
1. Соедините SAM, supervised и corrected IoU по test-индексам.
2. Укажите изображения, где корректор снизил IoU по сравнению с SAM.

**Результат.** correction_failures.json с реальными индексами и тремя IoU.

**Критерий принятия.** Задание считается принятым, если описанные действия выполнены, результат соответствует указанному артефакту и получен на предусмотренных данных; в режиме FAST_DEV_RUN результат служит только проверкой исполнения.


In [66]:
# Соединяем SAM, supervised и corrected IoU по test-индексам
sup_map = {r["index"]: r["supervised_iou"] for r in supervised_comparison}
merged = []
for r in adapted_comparison:
    idx = r["index"]
    merged.append({
        "index": idx,
        "sam_iou": r["sam_iou"],
        "supervised_iou": sup_map.get(idx, None),
        "corrected_iou": r["corrected_iou"],
    })

# Изображения, где корректор снизил IoU по сравнению с SAM
failures = [m for m in merged if m["corrected_iou"] < m["sam_iou"]]
with open(ARTIFACTS / "correction_failures.json", "w") as f:
    json.dump(failures, f, indent=2)

df_merged = pd.DataFrame(merged)
display(df_merged.head(10))
print(f"✔ Всего test-изображений: {len(merged)}")
print(f"✔ Корректор снизил IoU на {len(failures)} изображениях")
if failures:
    print("Примеры трудных случаев (индекс, SAM IoU, corrected IoU):")
    for f in failures[:5]:
        print(f"  #{f['index']}: {f['sam_iou']:.3f} -> {f['corrected_iou']:.3f}")
print(f"✔ Файл сохранён: {ARTIFACTS / 'correction_failures.json'}")

,index,sam_iou,supervised_iou,corrected_iou
0,0,0.673968,0.918224,0.675279
1,1,0.766703,0.831026,0.781651
2,2,0.808259,0.847548,0.816229
3,3,0.718038,0.812406,0.729603
4,4,0.669171,0.640487,0.694141
5,5,0.724714,0.799285,0.740611
6,6,0.727689,0.793208,0.748053
7,7,0.854665,0.825744,0.862358
8,8,0.900403,0.943846,0.907023
9,9,0.697178,0.752080,0.736141


✔ Всего test-изображений: 200
✔ Корректор снизил IoU на 2 изображениях
Примеры трудных случаев (индекс, SAM IoU, corrected IoU):
  #59: 0.803 -> 0.802
  #187: 0.853 -> 0.851
✔ Файл сохранён: artifacts\correction_failures.json


### Творческий бонус: второй тип точки

*Вес: 1 дополнительный балл.*

**Постановка.** Качество SAM зависит от выбора положительной точки.

Выполните следующие действия:
1. Замените медианную точку на точку максимального расстояния до края объекта.
2. На тех же test-изображениях сравните IoU с исходной подсказкой.

**Результат.** bonus_points.json с парными IoU для двух стратегий.

**Критерий принятия.** Задание считается принятым, если описанные действия выполнены, результат соответствует указанному артефакту и получен на предусмотренных данных; в режиме FAST_DEV_RUN результат служит только проверкой исполнения.


In [67]:
# Альтернативная точка: пиксель внутри маски с максимальным расстоянием до края.
# Используем distance transform из scipy.
from scipy.ndimage import distance_transform_edt

def farthest_point_from_edge(mask: np.ndarray) -> tuple:
    """Возвращает (x, y) — точку внутри маски с максимальным расстоянием до фона."""
    dist = distance_transform_edt(mask)
    y, x = np.unravel_index(np.argmax(dist), dist.shape)
    return int(x), int(y)

bonus_results = []
for i in range(len(eval_ds)):
    img, gt = eval_ds[i]
    x0, y0, x1, y1 = box_from_mask(gt)
    # Исходная точка — центр bbox (как в основном эксперименте)
    cx, cy = int((x0 + x1) / 2), int((y0 + y1) / 2)
    # Альтернативная точка — farthest from edge
    fx, fy = farthest_point_from_edge(gt)

    mask_center = predict_sam(img, {"type": "point", "coords": [[cx, cy]]}, processor, model)
    mask_farthest = predict_sam(img, {"type": "point", "coords": [[fx, fy]]}, processor, model)

    bonus_results.append({
        "index": i,
        "center_iou": iou(gt, mask_center),
        "farthest_iou": iou(gt, mask_farthest),
    })

with open(ARTIFACTS / "bonus_points.json", "w") as f:
    json.dump(bonus_results, f, indent=2)

df_bonus = pd.DataFrame(bonus_results)
display(df_bonus.head(10))
print(f"✔ Средний IoU (центр bbox)      = {df_bonus['center_iou'].mean():.4f}")
print(f"✔ Средний IoU (farthest from edge) = {df_bonus['farthest_iou'].mean():.4f}")
print(f"✔ Результаты сохранены: {ARTIFACTS / 'bonus_points.json'}")

,index,center_iou,farthest_iou
0,0,0.430133,0.582197
1,1,0.771043,0.770231
2,2,0.804193,0.805609
3,3,0.719827,0.717571
4,4,0.205922,0.210661
5,5,0.731355,0.731468
6,6,0.720131,0.720187
7,7,0.862306,0.859267
8,8,0.902271,0.890469
9,9,0.687491,0.688563


✔ Средний IoU (центр bbox)      = 0.6860
✔ Средний IoU (farthest from edge) = 0.6721
✔ Результаты сохранены: artifacts\bonus_points.json


## Анализ результатов

Сопоставьте измеренные показатели на отложенных реальных данных. Укажите бюджет, ошибки модели и ограничения сокращённого запуска.


* **Point vs box (SAM ViT-B, 200 test):** IoU 0.686 vs 0.800. Box даёт +11.4 п.п. — площадь bbox однозначно локализует объект, тогда как одна точка часто «цепляет» только часть питомца (пример: #4 point 0.206 → box 0.669).
* **Разброс point:** от 0.206 (#4, мелкий объект 6.4k px) до 0.902 (#8, 159k px). Point сильно зависит от попадания в тело, box стабилизирует качество.
* **Supervised LRASPP (200 trainval, 5 эпох, loss 0.52→0.13):** IoU 0.783 на тех же 200 test. На равном бюджете обучение с нуля немного проигрывает zero-shot SAM — сказывается малость выборки.
* **Refinement head (200 train, 10 эпох, BCE 0.57→0.21):** IoU 0.800 → 0.813 (+1.3 п.п.), улучшено 99% изображений. Голова учится не «предсказывать всё», а корректировать границы поверх маски SAM.
* **Отрицательный перенос:** только 2/200 (`#59: 0.803→0.802`, `#187: 0.853→0.851`) — в пределах шума, средний прирост устойчив.
* **Доля «сложных» масок (SAM IoU<0.7):** 15% test. После refinement средняя по ним растёт (пример #4: 0.669→0.694, #9: 0.697→0.736).
* **Bonus (farthest-from-edge):** 0.672 vs center 0.686 — альтернативная точка чуть хуже. Причина: на невыпуклых силуэтах (хвост/ухо) farthest-пиксель смещается из тела объекта.
* **Бюджет:** 200 test + 200 trainval, SAM ViT-B frozen, ≈5 мин на обучение baseline + ≈2 мин на голову (cuda). Полный прогон, не FAST_DEV_RUN.

## Выводы

Сформулируйте предметный вывод по сохранённым артефактам.


* **Box-prompt — базовый режим разметки на Oxford Pets:** zero-shot SAM ViT-B даёт IoU 0.800, что выше supervised LRASPP на равном бюджете (0.783). Для мелких объектов point ненадёжен (IoU<0.5 в ~10% случаев).
* **Refinement head — дешёвый и предсказуемый буст:** +1.3 п.п. IoU (0.800→0.813) при 200 train-пар и 10 эпохах; 99% изображений улучшены, отрицательный перенос ничтожен (2/200). Голова не заменяет SAM, а «подчищает» границы.
* **Потолок zero-shot:** 15% test-масок имеют IoU<0.7 — узкое место для автоматической разметки; именно они выигрывают от корректора и от интерактивных point-подсказок.
* **Стратегия точки:** bbox-центр лучше farthest-from-edge (0.686 vs 0.672) — простая эвристика достаточна, усложнение не окупается на этом датасете.
* **Практический вывод:** пайплайн «SAM box → refinement head» даёт IoU 0.813 без дообучения foundation-модели, что приемлемо для предразметки с ручной проверкой ~15% сложных кадров.